In [1]:
import argparse
import os
import pathlib
import sys
import uuid

import duckdb
import pandas as pd
from cytotable import convert, presets
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from parsl.config import Config
from parsl.executors import HighThroughputExecutor

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C5-1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)
dest_datatype = "parquet"

In [4]:
# show the tables
with duckdb.connect(input_sqlite_file) as con:
    tables = con.execute("SHOW TABLES").fetchdf()
    print(tables)
    nuclei_table = con.sql("SELECT * FROM Nuclei").df()
    cells_table = con.sql("SELECT * FROM Cell").df()
    cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
    organoid_table = con.sql("SELECT * FROM Organoid").df()
    nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()

            name
0           Cell
1      Cytoplasm
2         Nuclei
3  Nucleocentric
4       Organoid


In [5]:
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())
# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)
# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# connect to DuckDB and register the tables
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

## Reorder object IDs

In [7]:
# replace the object_id with a new unique ID
organoid_table["object_id"] = [i for i in range(1, organoid_table.shape[0] + 1)]
merged_df["object_id"] = [i for i in range(1, merged_df.shape[0] + 1)]
nucleocentric_table["object_id"] = [
    i for i in range(1, nucleocentric_table.shape[0] + 1)
]

In [8]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (3, 3961)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C5-1,21871111.0,881.712573,624.071282,26.662352,31215198.0,494,1271,244,...,6.225227e-322,8.152083e-322,1.007894e-321,1.200580e-321,1.393265e-321,1.585951e-321,1.778636e-321,1.971322e-321,2.164008e-321,2.356693e-321
1,2,C5-1,120172.0,290.525422,760.369287,6.998544,169000.0,223,348,710,...,6.274634e-322,8.201490e-322,1.012835e-321,1.205520e-321,1.398206e-321,1.590891e-321,1.783577e-321,1.976263e-321,2.168948e-321,2.361634e-321
2,3,C5-1,190924.0,419.598709,864.783348,7.306122,256956.0,353,486,796,...,2.863360e+02,3.139670e+02,2.867346e+02,3.146183e+02,2.871789e+02,2.872635e+02,2.866362e+02,3.144529e+02,2.866812e+02,2.872574e+02


In [9]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (62, 11883)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_ER_Texture_Variance-3-03-256,Cytoplasm_ER_Texture_Variance-3-04-256,Cytoplasm_ER_Texture_Variance-3-05-256,Cytoplasm_ER_Texture_Variance-3-06-256,Cytoplasm_ER_Texture_Variance-3-07-256,Cytoplasm_ER_Texture_Variance-3-08-256,Cytoplasm_ER_Texture_Variance-3-09-256,Cytoplasm_ER_Texture_Variance-3-10-256,Cytoplasm_ER_Texture_Variance-3-11-256,Cytoplasm_ER_Texture_Variance-3-12-256
0,1,C5-1,70405.0,999.175982,547.139053,3.247980,95832.0,935,1056,499,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,C5-1,23072.0,196.618889,217.779690,2.973734,35287.0,163,234,183,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,C5-1,93352.0,827.655144,548.871990,5.589789,137196.0,773,884,498,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,C5-1,71500.0,882.212881,801.368378,7.165329,106275.0,827,936,764,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,C5-1,73486.0,1005.557263,821.117968,7.119955,107900.0,955,1055,780,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (62, 1538)


,object_id,image_set,Nucleocentric_ER_SAMMed3D_Feature0,Nucleocentric_ER_SAMMed3D_Feature1,Nucleocentric_ER_SAMMed3D_Feature10,Nucleocentric_ER_SAMMed3D_Feature100,Nucleocentric_ER_SAMMed3D_Feature101,Nucleocentric_ER_SAMMed3D_Feature102,Nucleocentric_ER_SAMMed3D_Feature103,Nucleocentric_ER_SAMMed3D_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature90,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99
0,1,C5-1,-0.309459,-0.164567,0.115346,0.042141,-0.075798,0.225040,-0.033044,-0.121703,...,-0.007193,-0.086312,0.030703,-0.010622,0.018842,-0.030385,-0.061651,0.246042,0.391427,0.214142
1,2,C5-1,-0.508843,-0.181947,0.142647,-0.012031,0.039214,0.102439,-0.008175,-0.062324,...,-0.007701,-0.042624,0.121502,-0.010894,0.024822,-0.043626,-0.014368,0.213620,0.317915,0.171356
2,3,C5-1,-0.067428,-0.183855,0.058745,-0.079168,0.052330,0.360698,0.012013,-0.078692,...,-0.007429,-0.066249,0.031035,-0.010470,0.009976,-0.010346,-0.045585,0.231124,0.381545,0.162618
3,4,C5-1,-0.235951,-0.099007,0.019764,-0.070681,-0.093577,0.210878,-0.012840,-0.089681,...,-0.008198,-0.082786,0.031806,-0.010532,0.004213,-0.017372,-0.076118,0.235571,0.395263,0.146230
4,5,C5-1,0.095864,-0.106371,-0.001354,0.020755,-0.015388,0.305872,0.150738,-0.163737,...,-0.005944,-0.027947,0.102712,-0.010457,0.026607,-0.003603,0.009239,0.285701,0.328337,0.216941
